In [0]:
USE CATALOG adb_classic_compute_catalog;
USE SCHEMA gold;

CREATE OR REPLACE VIEW adb_classic_compute_catalog.gold.assignment_activity_governed_vw AS
SELECT
  g.assignment_id,
  g.employee_id,

  CASE
    WHEN is_member('account users') THEN
      regexp_replace(e.user_principal_name, '(^.).*(@.*$)', '$1***$2')
    ELSE NULL
  END AS masked_user_principal_name,

  CASE
    WHEN is_member('account users') THEN NULL
    ELSE e.user_principal_name
  END AS full_user_principal_name_for_admin_review,

  g.application_id,
  g.license_tier_id,
  g.department_id,
  g.region_code,
  g.employment_status,
  g.activity_status,

  CASE
    WHEN g.activity_status = 'recent_qualifying_activity' THEN 'Recently used'
    WHEN g.activity_status = 'older_qualifying_activity' THEN 'Used, but not recently'
    WHEN g.activity_status = 'sign_in_only_activity' THEN 'Signed in only'
    WHEN g.activity_status = 'no_observed_activity' THEN 'No observed activity'
    WHEN g.activity_status = 'terminated_employee_review' THEN 'Terminated employee review'
    ELSE 'Needs review'
  END AS activity_status_label,

  CASE
    WHEN g.activity_status = 'terminated_employee_review' THEN 1
    ELSE 0
  END AS requires_access_review,


  g.latest_qualifying_activity_ts,
  g.latest_sign_in_ts,
  g.qualifying_event_count,
  g.sign_in_event_count,

  current_timestamp() AS served_at_utc
FROM adb_classic_compute_catalog.gold.assignment_activity_status AS g
LEFT JOIN adb_classic_compute_catalog.silver.employee_snapshot_valid_dlt AS e
  ON g.employee_id = e.employee_id
WHERE
  is_member('account users');

GRANT USE CATALOG ON CATALOG adb_classic_compute_catalog TO `account users`;

GRANT USE SCHEMA ON SCHEMA adb_classic_compute_catalog.gold TO `account users`;

GRANT SELECT ON VIEW adb_classic_compute_catalog.gold.assignment_activity_status_serving_vw TO `account users`;

GRANT SELECT ON VIEW adb_classic_compute_catalog.gold.assignment_activity_kpi_vw TO `account users`;

GRANT SELECT ON TABLE adb_classic_compute_catalog.gold.assignment_activity_kpi_snapshot TO `account users`;

GRANT SELECT ON VIEW adb_classic_compute_catalog.gold.assignment_activity_governed_vw TO `account users`;